In [1]:
import pandas as pd
import random
import os
import csv

# Paths
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
INPUT_FILE = os.path.join(PROCESSED_DIR, "01_MASTER_Original_12k_V2.csv")

# Load V2 data
print("Loading V2 Master Dataset...")
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=['original_comment']).reset_index(drop=True)

# Calculate word count based on FULL content (ignores newlines automatically)
df['word_count'] = df['original_comment'].apply(lambda x: len(str(x).split()))

# 1. Word Jaccard (Topic Distortion Proof)
def word_jaccard(str1, str2):
    # Splits by any whitespace/newline, ensuring full multi-line content is checked
    set1 = set(str(str1).lower().split())
    set2 = set(str(str2).lower().split())
    if not set1 or not set2:
        return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# 2. Bigram Jaccard (Grammar Distortion Proof)
def get_bigrams(words):
    return set(zip(words[:-1], words[1:]))

def bigram_jaccard(words1, words2):
    if len(words1) < 2 or len(words2) < 2:
        return 0.0
    b1 = get_bigrams(words1)
    b2 = get_bigrams(words2)
    if not b1 or not b2:
        return 0.0
    return len(b1.intersection(b2)) / len(b1.union(b2))

print("Setup complete. Data loaded and validated.")

Loading V2 Master Dataset...
Setup complete. Data loaded and validated.


In [2]:
print("Generating Distractor A (Topic Swap: Word & Token Matched)... Please wait.")

distractor_a_comments =[]

# Figure out the exact column name for tokens (handles V1/V2 differences)
token_col = 'code_token_length' if 'code_token_length' in df.columns else 'token_count'

for index, row in df.iterrows():
    orig_comment = str(row['original_comment'])
    target_wc = row['word_count']
    target_tc = row[token_col]
    
    # Define a 15% tolerance window for the code token length (min 5 tokens)
    tc_margin = max(5, int(target_tc * 0.15))
    
    # 1. Strict Filter: Match Comment Length (+/- 2 words) AND Code Token Length (+/- 15%)
    pool_df = df[
        (df['word_count'].between(max(1, target_wc - 2), target_wc + 2)) &
        (df[token_col].between(max(1, target_tc - tc_margin), target_tc + tc_margin)) &
        (df['original_comment'] != orig_comment)
    ]
    
    # 2. Relaxed Filter: If strict matching yields < 5 candidates, drop the token constraint
    if len(pool_df) < 5:
        pool_df = df[
            (df['word_count'].between(max(1, target_wc - 2), target_wc + 2)) &
            (df['original_comment'] != orig_comment)
        ]
        
    candidate_pool = pool_df['original_comment'].tolist()
    random.shuffle(candidate_pool)
    
    found = False
    for candidate in candidate_pool:
        # FULL CONTENT CHECK: Ensure < 30% vocabulary overlap
        if word_jaccard(orig_comment, candidate) < 0.3:
            distractor_a_comments.append(candidate)
            found = True
            break
            
    # 3. Absolute Fallback (Highly unlikely to trigger)
    if not found:
        while True:
            rand_comment = df['original_comment'].sample(n=1).iloc[0]
            if word_jaccard(orig_comment, rand_comment) < 0.3:
                distractor_a_comments.append(rand_comment)
                break

    # Print progress so you know it's working
    if (index + 1) % 2000 == 0:
        print(f"Processed {index + 1} / {len(df)} records...")

df['distractor_a'] = distractor_a_comments
print("Distractor A successfully generated with dual-constraint matching!")

Generating Distractor A (Topic Swap: Word & Token Matched)... Please wait.
Processed 2000 / 12000 records...
Processed 4000 / 12000 records...
Processed 6000 / 12000 records...
Processed 8000 / 12000 records...
Processed 10000 / 12000 records...
Processed 12000 / 12000 records...
Distractor A successfully generated with dual-constraint matching!


In [3]:
print("Generating Distractor B (Shuffled Grammar)... Please wait.")

distractor_b_comments =[]

for index, row in df.iterrows():
    # Split the full multi-line comment into a list of words
    words = str(row['original_comment']).split()
    
    if len(words) <= 2:
        # Can't deeply shuffle 1 or 2 words, so just reverse them
        words.reverse()
        distractor_b_comments.append(" ".join(words))
        continue
        
    best_shuffle = words[:]
    
    # Try up to 15 times to break the grammar
    for _ in range(15):
        random.shuffle(best_shuffle)
        # Check if the sequence of words (Bigrams) is < 10% similar to the original
        if bigram_jaccard(words, best_shuffle) < 0.1:
            break
            
    # Rejoin into a string
    distractor_b_comments.append(" ".join(best_shuffle))

df['distractor_b'] = distractor_b_comments
print("Distractor B successfully generated.")

Generating Distractor B (Shuffled Grammar)... Please wait.
Distractor B successfully generated.


In [4]:
import csv

print("Formatting and saving final V2 datasets...")

# 1. Format Original 12k (Label 1)
df_orig = df.drop(columns=['distractor_a', 'distractor_b', 'word_count']).copy()
df_orig.rename(columns={'original_comment': 'comment'}, inplace=True)
df_orig['label'] = 1

# 2. Format Distractor A 12k (Label 0)
df_dist_a = df.drop(columns=['original_comment', 'distractor_b', 'word_count']).copy()
df_dist_a.rename(columns={'distractor_a': 'comment'}, inplace=True)
df_dist_a['label'] = 0

# 3. Format Distractor B 12k (Label 0)
df_dist_b = df.drop(columns=['original_comment', 'distractor_a', 'word_count']).copy()
df_dist_b.rename(columns={'distractor_b': 'comment'}, inplace=True)
df_dist_b['label'] = 0

# Save files with strict quoting to preserve \n in the code
file_orig = os.path.join(PROCESSED_DIR, "01_Original_12k_V2.csv")
file_a = os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k_V2.csv")
file_b = os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k_V2.csv")

df_orig.to_csv(file_orig, index=False, quoting=csv.QUOTE_ALL)
df_dist_a.to_csv(file_a, index=False, quoting=csv.QUOTE_ALL)
df_dist_b.to_csv(file_b, index=False, quoting=csv.QUOTE_ALL)

print("\n--- SAVE COMPLETE ---")
print(f"Original saved to: {file_orig}")
print(f"Distractor A saved to: {file_a}")
print(f"Distractor B saved to: {file_b}")

Formatting and saving final V2 datasets...

--- SAVE COMPLETE ---
Original saved to: C:\Users\HP\Desktop\thesis_preprocessing\data\processed\01_Original_12k_V2.csv
Distractor A saved to: C:\Users\HP\Desktop\thesis_preprocessing\data\processed\02_Distractor_A_Swapped_12k_V2.csv
Distractor B saved to: C:\Users\HP\Desktop\thesis_preprocessing\data\processed\03_Distractor_B_Shuffled_12k_V2.csv
